# Zerlaut parameter exploration

## Initialization

**Environment**

In [ ]:
from dotenv import dotenv_values, find_dotenv
from pathlib import Path
import os, sys

# Busqueda de ficheros dotenv
defaults_file = find_dotenv('.env_default', usecwd=True)

if not defaults_file:
    raise FileNotFoundError('Start Jupyter within the repository containing .env_default.')
env_dir = Path(defaults_file).resolve().parent

# Actualización de varaibles de entorno
settings = {**dotenv_values(defaults_file), **dotenv_values(env_dir / '.env')}
settings = {k: v if v is not None else '' for k, v in settings.items() }

os.environ.update(settings)

# Librerías auxiliares
import_dir = Path(settings.get('IMPORT_DIR') or env_dir).expanduser()
if not import_dir.is_absolute():
    import_dir = env_dir / import_dir
sys.path.insert(0, str(import_dir.resolve()))

**Modules**

In [ ]:
import json5 as json
from copy import deepcopy

import numpy as np
from scipy.stats import qmc
from scipy.stats import beta, truncnorm, uniform

import matplotlib.pyplot as plt
import pandas as pd

from IPython.display import display
# from itables import show
# from ipydatagrid import DataGrid
from ipyaggrid import Grid


In [ ]:
from aux.environment import resolve_directory

from aux.tvb.stimulus import build_stimulus
# from aux.tvb.results import SimulationMetadata, ModelMetadata, ResultMetadata
# from aux.tvb.results import save_single_region_simulation
# from aux.tvb.results import load_single_region_simulation
# from aux.tvb.plotting import plot_results

from aux.tvb.model import build_model
from aux.tvb.integrator import build_integrator
from aux.tvb.simulator import build_single_region_simulator
from aux.tvb.model import get_model_class

**Console**

In [ ]:
from rich.console import Console
from rich.table import Table

pprint = Console(force_terminal=True, force_jupyter=False, width=140).print

pprint(f'The console is "ready" to print messages with rich formatting.')

**Set main directory paths**

In [ ]:
WORK_DIR = resolve_directory(settings['WORK_DIR'], env_dir)

os.makedirs(WORK_DIR, exist_ok=True)
pprint(f'Working directory: {WORK_DIR}')

## Auxiliar Functions

### Create and show tables

In [ ]:
def make_parameter_table(params, groups):
    group_lookup = {
        name: group
        for group, names in groups.items()
        for name in names
    }
    return pd.DataFrame([
        {'Parameter': name, **data, 'Group': group_lookup.get(name)}
        for name, data in params.items()
    ])

def show_table(df):
   display(Grid(
        grid_data=df,
        theme="ag-theme-balham-dark",
        grid_options={
            "defaultColDef": {"filter": True, "resizable": True, 'editable': False,},
            #'columnDefs': [],
            'pagination': True,
            'paginationPageSize': 50,
        },
        quick_filter=True,
        columns_fit="auto"
    ))

### Sampling

In [ ]:
# Allow appling sampling functions to parameters with list values, 
# by flattening the parameter dictionary and restoring it after sampling.

def flatten_params(params):
    flat = {}
    for name, config in params.items():
        first = next(iter(config.values()))
        if isinstance(first, list):
            for i in range(len(first)):
                flat[f"{name}*{i:02}"] = {k: v[i] for k, v in config.items()}
        else:
            flat[name] = config
    return flat


def restore_flat_params(flat_params):
    result = {}
    for name, config in sorted(flat_params.items()):
        base, sep, _ = name.partition("*")
        if not sep:
            result[name] = config
        else:
            if isinstance(config, dict):
                group = result.setdefault(base, {})
                for k, v in config.items():
                    group.setdefault(k, []).append(v)
            else:
                result.setdefault(base, []).append(config)
    return result


In [ ]:
# Add sampling configuration to parameters, for truncated normal and beta distributions.

def add_truncnorm_config(params, std_factor=0.2, names=None):
    result = deepcopy(params)

    selected_names = result if names is None else names

    for name in selected_names:
        config = result[name]
        config["std"] = (config["max"] - config["min"]) * std_factor
        config["method"] = "truncnorm"

    return result

def add_beta_config(params, lambda_, names=None):
    result = deepcopy(params)

    selected_names = result if names is None else names

    for name in selected_names:
        config = result[name]
        config["lambda"] = lambda_
        config["method"] = "beta"

    return result

In [ ]:
# Configure scalar and list-valued parameters without modifying the input.
def add_beta_config_safe(params, lambda_, names=None):
    flat = flatten_params(params)
    return restore_flat_params(add_beta_config(flat, lambda_=lambda_, names=names))


def add_truncnorm_config_safe(params, std_factor=0.2, names=None):
    flat = flatten_params(params)
    return restore_flat_params(add_truncnorm_config(flat, std_factor=std_factor, names=names))


In [ ]:
def _uniform_sample(s, minimum, maximum):
    scale= maximum - minimum
    return minimum + s * scale, 1/scale


def _truncnorm_sample(s, minimum, maximum, peak, std):
    a = (minimum - peak) / std
    b = (maximum - peak) / std
    dist = truncnorm(a, b, loc=peak, scale=std)
    return (value := dist.ppf(s)), dist.pdf(value)


def _beta_sample(s, minimum, maximum, peak, lam):
    relative_peak = (peak - minimum) / (maximum - minimum)
    a = 1 + relative_peak * lam
    b = 1 + (1 - relative_peak) * lam
    dist = beta(a, b, loc=minimum, scale=maximum - minimum)
    return (value := dist.ppf(s)), dist.pdf(value)

def _sample_from_pdf(s, param):
    minimum, maximum = param["min"], param["max"]

    if minimum == maximum:
        return minimum, 1.0

    match param.get("method", "uniform"):
        case "truncnorm":
            return _truncnorm_sample(
                s, minimum, maximum, param["default"], param["std"]
            )
        case "beta":
            return _beta_sample(
                s, minimum, maximum, param["default"], param["lambda"]
            )
        case _:
            return _uniform_sample(s, minimum, maximum)


def sample_params(params, n_samples, fordware=0, seed=42):
    sampler = qmc.Sobol(d=len(params), scramble=True, seed=seed)
    if fordware > 0:
        sampler.fast_forward(fordware)

    for s in sampler.random(n_samples):
        values, densities = {}, {}
        for i, (name, config) in enumerate(params.items()):
            value, density = _sample_from_pdf(s[i], config)
            values[name] = value
            densities[name] = density

        yield values, densities

        # # Mas elegante, pero menos eficiente
        # values, densities = zip(*map(_sample_from_pdf, s, params.values()))
        # yield dict(zip(params, values)), dict(zip(params, densities))

### Parameter exploration

In [ ]:
def make_grid_samples(params, names, samples_per_axis=4,
                       spacing='linear', integer_parameters=frozenset(),
                       seed=42,
                       first_index=0, second_index=0): # estos dos parametros solo se usan si los dos parametros de muestreo regular son listas
    
    # Paso 1
    # Los dos primereros parametros se exploran de forma regular, el resto se muestrea con sobol.
    sampling_params = {name: params[name] for name in names}
    axes, axis_keys = {}, {}
    for name, index in zip(names[:2], (first_index, second_index)):
        
        config = sampling_params[name]
        is_list = isinstance(config['default'], list)
        minimum = config['min'][index] if is_list else config['min']
        maximum = config['max'][index] if is_list else config['max']
        default = config['default'][index] if is_list else config['default']

        axis_keys[name] = f'{name}*{index:02}' if is_list else name

        if spacing == 'linear':
            #axes[name] = np.linspace(minimum, maximum, samples_per_axis)
            # Prefiero muestrea en el centro de los intervalos
            step = (maximum - minimum) / samples_per_axis
            axes[name] = minimum + (np.arange(samples_per_axis) + 0.5) * step
        
        ## Escala logaritmica, Ojo el primer valor no puede ser 0
        else:
            # Si el mínimo es 0, usamos un valor muy pequeño
            start = default*0.001 if minimum == 0 else minimum
            start += 1e-5 if start <= 0 else 0 # si default es 0
            axes[name] = np.geomspace(start, maximum, samples_per_axis)

    
    # Paso 2
    # Numero de ejecuciones y de columnas,    
    n_rows = samples_per_axis if len(names) > 1 else 1
    n_runs = n_rows * samples_per_axis

    #Paso 3
    # El resto de parametros se muestran con sobol.
    # comprobar si ha parametros extra
    flat = flatten_params(sampling_params)
    extra = {key: config for key, config in flat.items() if key not in axis_keys.values()}

        
    if extra:
        sobol_rows = [
            values for values, _ in sample_params(extra, n_samples=n_runs, seed=seed)
        ]
    else: 
        sobol_rows = [{} for _ in range(n_runs)]

    # Paso 4
    #Valores por defecto de todos los parametros
    defaults = {
        name: config['default']
        for name, config in params.items()
    }

    # Paso 5
    # Las muestras estan en axes[name] y sobol_rows[i]
    grid = []
    for row in range(n_rows):
        grid_row = []
        for column in range(samples_per_axis):
            # Parametros muestreados con sobol, si los hay
            sampled = dict(sobol_rows[row * samples_per_axis + column])

            # Se añade el muestro regular.
            for position, name in enumerate(names[:2]):
                location = row if position == 0 and n_rows > 1 else column
                sampled[axis_keys[name]] = float(axes[name][location])

            # Restaurar los valores muestreados
            sampled_values = restore_flat_params(sampled)

            # Ajustar los valores de los parametros enteros
            for name in integer_parameters & sampled_values.keys():
                value = sampled_values[name]
                sampled_values[name] = (
                    [int(round(item)) for item in value]
                    if isinstance(value, list) else int(round(value))
                )

            # El grid final combina los parametro por defecto con los muestreados
            # ToDo: Comprobar si el deepcopy es necesario, quizas no lo sea.
            grid_row.append(deepcopy(defaults) | sampled_values)
        grid.append(grid_row)
    
    return grid

In [ ]:
def show_sample_table(grid, names):
    table = Table(title=f'{sum(map(len, grid))} parameter samples', show_lines=True)
    table.add_column('i / j', justify='right')
    for j in range(len(grid[0])):
        table.add_column(f'j={j}')
    for i, row in enumerate(grid):
        cells = []
        for values in row:
            lines = []
            for name in names:
                value = values[name]
                text = str(value) if isinstance(value, list) else f'{value:.6g}'
                lines.append(f'{name}: {text}')
            cells.append('\n'.join(lines))
        table.add_row(str(i), *cells)
    pprint(table)

In [ ]:
def simulate_grid(grid, stimulus, time_ms, # *,
                  model_name, output_states,
                  integer_parameters, dt_ms, integrator_name,
                  noise_config, noise_seed):
    results = []
    for i, row in enumerate(grid):
        row_results = []
        for j, parameters in enumerate(row):
            pprint(f'Running simulation: {i}, {j}')
            model = build_model(
                parameters, integer_parameters,
                model_class=model_name, variable_of_interest=output_states,
            )
            integrator = build_integrator(
                dt_ms, integrator_class=integrator_name,
                noise_config=noise_config,
            )
            simulator = build_single_region_simulator(model, stimulus, integrator)
            integrator.noise.random_stream.seed(noise_seed)
            time_sim, raw = simulator.run(n_steps=len(time_ms))[0]
            row_results.append((time_sim, raw[:, 0, 0, 0]))
        results.append(row_results)

    return results


In [ ]:
def plot_e_grid(grid, results, names, stimulus, first_index=0, second_index=0, log_scale=False):
    names = names[:2]
    indices = (first_index, second_index)
    n_rows, n_columns = len(grid), len(grid[0])
    fig, axes = plt.subplots(
        n_rows, n_columns, figsize=(4 * n_columns, 2.8 * n_rows),
        sharex=True, sharey=True, squeeze=False, constrained_layout=True,
    )

    for i, row in enumerate(grid):
        for j, parameters in enumerate(row):
            time_sim, e_values = results[i][j]
            ax = axes[i, j]
            ax.plot(time_sim, e_values, label='E')
            for key, values in stimulus:
                ax.plot(time_ms, values[:, 0], '--',
                        alpha=0.7, label=key)
            ax.set_title('; '.join(
                f'{name}[{index}]={parameters[name][index]:.3g}'
                if isinstance(parameters[name], list)
                else f'{name}={parameters[name]:.3g}'
                for name, index in zip(names, indices)
            ))
            
            if i == n_rows - 1:
                ax.set_xlabel('Time (ms)')
            if j == 0:
                ax.set_ylabel('Out / stimulus (kHz)')
            if i == 0 and j == 0:
                ax.legend(fontsize='small')

    if log_scale:
        for ax in axes.flat:
            ax.set_yscale('symlog', linthresh=1e-3)
            ax.axhline(y=0, color='black', linewidth=0.8)
            ax.axhline(y=0.01, color='red', linewidth=0.8)
            ax.axhline(y=0.001, color='yellow', linewidth=0.8)
    fig.suptitle('Out and stimulus')
    plt.show()
    return fig, axes


### <font color=#fa0>Functions to adjust parameter ranges</font>

Especialmente relevantes en para los parámetros de $P_e$ y $P_i$ que carecen de rango

In [ ]:
def _adjust_value(value, margin, sign):
    if isinstance(value, list):
        return [v + sign * abs(v) * margin for v in value]
    return value + sign * abs(value) * margin


def bounds_from_default(params, names=None, margin=0.10):
    result = deepcopy(params)
    selected_names = result if names is None else names

    for name in selected_names:
        config = result[name]
        default = config["default"]

        config["min"] = _adjust_value(default, margin, -1)
        config["max"] = _adjust_value(default, margin, 1)

    return result

In [ ]:
def expand_bounds(params, names=None, margin=0.10):
    if margin < 0:
        raise ValueError("Variation must be non-negative.")

    result = deepcopy(params)
    selected_names = result if names is None else names

    for name in selected_names:
        config = result[name]

        config["min"] = _adjust_value(config["min"], margin, -1)
        config["max"] = _adjust_value(config["max"], margin, 1)

    return result



## Load parameters and groups

In [ ]:
CONFIG_PATH = WORK_DIR / 'config' / 'zerlaut' / 'params_raw.json'

pprint(f'Parameter configuration: {CONFIG_PATH}')

In [ ]:
GROUPS = {
    'Membrane properties': ['g_L', 'E_L_e', 'E_L_i', 'C_m'],
    'Adaptation': ['b_e', 'a_e', 'b_i', 'a_i', 'tau_w_e', 'tau_w_i'],
    'Synaptic properties': ['E_e', 'E_i', 'Q_e', 'Q_i', 'tau_e', 'tau_i'],
    'Network properties': ['N_tot', 'p_connect_e', 'p_connect_i', 'g', 'K_ext_e', 'K_ext_i', 'T'],
    'Phenomenological threshold coefficients': ['P_e', 'P_i'],
    'External inputs': [
        'external_input_ex_ex', 'external_input_ex_in',
        'external_input_in_ex', 'external_input_in_in',
    ],
    'Noise': ['tau_OU', 'weight_noise'],
    'Inhibitory coupling scaling': ['S_i'],
}

In [ ]:
with CONFIG_PATH.open(encoding='utf-8') as config_file:
    PARAMS = json.load(config_file)

pprint(f'Loaded {len(PARAMS)} parameter names in {len(GROUPS)} groups.')

**Show paramters**

In [ ]:
parameter_table = make_parameter_table(PARAMS, GROUPS)
print('Zerlaut parameters and their default values:')
#show(parameter_table)
# display(parameter_table)
# show(parameter_table, columnControl=["order", "searchDropdown"], autoWidth=False)
# display(DataGrid(parameter_table))
show_table(parameter_table)


**Parameter counts**

In [ ]:
parameter_counts = parameter_table['default'].map(
    lambda value: len(value) if isinstance(value, list) else 1
)
counts_by_group = parameter_counts.groupby(parameter_table['Group'], sort=False).sum()

pprint('Parameters per group:', counts_by_group.to_dict())
pprint(f'Total parameters: {parameter_counts.sum()}')
pprint(f"Excluding external inputs: {parameter_counts[parameter_table['Group'] != 'External inputs'].sum()}")
pprint(f"Excluding external inputs and noise: {parameter_counts[~parameter_table['Group'].isin(['External inputs', 'Noise'])].sum()}")

## Parameter test

### Common settings

In [ ]:
# 1.3) Parámetros del modelo Zerlaut-AdEx.
#MODEL_NAME = "ZerlautAdaptationFirstOrder"
#MODEL_NAME = "ZerlautAdaptationSecondOrder"
MODEL_NAME = "ZerlautAdaptationFirstOrder" # Solo por tiempo!!!!

BASE_PARAMS = bounds_from_default(PARAMS, ["P_e", "P_i"], margin=0.10)
# BASE_PARAMS = add_truncnorm_config_safe(BASE_PARAMS, std_factor=0.2)
# BASE_PARAMS = add_beta_config_safe(BASE_PARAMS, lambda_=2.0)
for key in ("external_input_ex_ex", "external_input_ex_in", "external_input_in_ex", "external_input_in_in"):
    BASE_PARAMS.pop(key,None) 


INTEGER_PARAMETERS = {"N_tot", "K_ext_e", "K_ext_i"} 

pprint('New bounded parameters:', str({k: v for k, v in BASE_PARAMS.items() if k in ["P_e", "P_i"]}))

In [ ]:
# 1) Entradas de la simulación.

# 1.0) Salida de la simulación.
# OUTPUT_STATES = None
# OUTPUT_STATES = ['E', 'I', 'W_e', 'W_i', 'ou_drift']
OUTPUT_STATES = ['E']

# Solo requerido para tareas como plotting.
STATE_UNITS = {
    "E": "kHz", "I": "kHz",
    "C_ee": "kHz²", "C_ei": "kHz²", "C_ii": "kHz²",
    "W_e": "pA", "W_i": "pA", "ou_drift": "N.A.",
}

STIMULUS_UNITS = {
    "external_input_ex_ex": "kHz",
    "external_input_ex_in": "kHz",
    "external_input_in_ex": "kHz",
    "external_input_in_in": "kHz",
}


In [ ]:
# 1.1) Parámetros de la simulación.
DURATION_MS = 2000.0    # Duración de la simulación en milisegundos.
DT_MS = 0.5             # Paso de integración en milisegundos.

zero_nsig_ = [0] * len(svar := get_model_class(MODEL_NAME).state_variables)
nsig_ = [int(i == svar.index("ou_drift")) for i in range(len(svar))]


INTEGRATOR_NAME = 'HeunStochastic'  
NOISE ={
    "name": "Additive", "params": {"nsig": nsig_} # [0,0,0,0,1] in first order model
}

NO_NOISE ={
    "name": "Additive", "params": {"nsig": zero_nsig_} # [0,0,0,0,0] in first order
}

NOISE_SEED = 42

In [ ]:
# 1.2) Parámetros de la entrada. En este caso un pulso.
STIMULUS = {
    "external_input_ex_ex": {
        "pulse_train": {
            "baseline_khz": 0.001,        # Frecuencia de disparo basal 1Hz
            "pulse_amplitude_khz": 0.01,  # Amplitud del pulso 10 Hz
            "pulse_onset_ms": 200.0,      # Instante de inicio del pulso
            "pulse_duration_ms": 160.0,   # Duración del pulso
            "pulse_period_ms": 320.0,     # Periodo del pulso
        },
    },   
    "external_input_in_ex": {"external_input_ex_ex": 1}, 
}


STIMULUS_EX = {
    "external_input_ex_ex": {
        "pulse_train": {
            "baseline_khz": 0.005,        # Frecuencia de disparo basal
            "pulse_amplitude_khz": 0.04,  # Amplitud del pulso
            "pulse_onset_ms": 200.0,      # Instante de inicio del pulso
            "pulse_duration_ms": 160.0,   # Duración del pulso
            "pulse_period_ms": 320.0,     # Periodo del pulso
        },
    },   
}




In [ ]:
time_ms, stimulus = build_stimulus(STIMULUS, DURATION_MS, DT_MS)
time_ms, stimulus_ex = build_stimulus(STIMULUS_EX, DURATION_MS, DT_MS)

# Plotting

fig, ax = plt.subplots(figsize=(10, 3))

for key, values in stimulus:
    ax.plot(time_ms, values, label=key)

ax.set_title("Stimulation Signals")
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Stimulus (KHz)")
ax.legend()
plt.show()

In [ ]:
# Sample parameters
SAMPLES_PER_AXIS = 4
GRID_SPACING = 'linear'  # Set to 'log' for logarithmic spacing.
SAMPLING_SEED = 42
FIRST_PARAM_INDEX = 0
SECOND_PARAM_INDEX = 0


### Noise

In [ ]:
PARAMS_TO_EXPLORE = GROUPS['Noise'] #+["whatever"]

pprint(f'Noise len: {len(GROUPS["Noise"])}')
pprint(f'Exploring parameters: {PARAMS_TO_EXPLORE}')
pprint(f'Default: {[BASE_PARAMS[p]["default"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Min: {[BASE_PARAMS[p]["min"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Max: {[BASE_PARAMS[p]["max"] for p in PARAMS_TO_EXPLORE]}')


In [ ]:
FIX_PARAMS = {"Q_e": 1.0} # Para reproducir los experimentos de otras cuadernos.
EXPLORATION_PARAMS = deepcopy(BASE_PARAMS)

#EXPLORATION_PARAMS["weight_noise"]["min"] = 0.0001
#EXPLORATION_PARAMS["weight_noise"]["max"] = 1.0
#EXPLORATION_PARAMS["tau_OU"]["min"] = 0.4

# Modificar los parámetros de muestro
# EXPLORATION_PARAMS |= add_beta_config_safe(
#     {name: EXPLORATION_PARAMS[name] for name in PARAMS_TO_EXPLORE},
#     lambda_=5
# )

GRID_SPACING = 'log' # 'linear'

USE_NOISE = True
USE_STIMULUS_EX = False

In [ ]:
grid = make_grid_samples(EXPLORATION_PARAMS, PARAMS_TO_EXPLORE,
                         samples_per_axis=SAMPLES_PER_AXIS, spacing=GRID_SPACING,
                         integer_parameters=INTEGER_PARAMETERS, seed=SAMPLING_SEED,
                         first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX)

# Parametros que has querido cambiar para la simulación
for i, grid_row in enumerate(grid):
    for j, parameters in enumerate(grid_row):
        grid[i][j] = parameters | FIX_PARAMS
    

In [ ]:
results = simulate_grid(
    grid, 
    stimulus if not USE_STIMULUS_EX else stimulus_ex, 
    time_ms, 
    model_name=MODEL_NAME,
    output_states=OUTPUT_STATES, 
    integer_parameters=INTEGER_PARAMETERS,
    dt_ms=DT_MS, 
    integrator_name=INTEGRATOR_NAME,
    noise_config=NOISE if USE_NOISE else NO_NOISE, 
    noise_seed=NOISE_SEED,
)

pprint(f'Completed {sum(map(len, results))} simulations.')

In [ ]:
for p in PARAMS_TO_EXPLORE:
    pprint(f'Parameter: {p}')
    pprint(f'  Default: {BASE_PARAMS[p]["default"]}')
    pprint(f'  Min: {BASE_PARAMS[p]["min"]}')
    pprint(f'  Max: {BASE_PARAMS[p]["max"]}')

show_sample_table(grid, list(PARAMS_TO_EXPLORE))

In [ ]:
pprint(f'Default: {dict((p, BASE_PARAMS[p]["default"]) for p in PARAMS_TO_EXPLORE)}')

fig, axes = plot_e_grid(
    grid, results, PARAMS_TO_EXPLORE,
    stimulus_ex if USE_STIMULUS_EX else stimulus
)
fig.show()

#### Plot noise

In [ ]:
EXPLORATION_PARAMS["weight_noise"]["min"] = 0.0001
EXPLORATION_PARAMS["weight_noise"]["max"] = 1.0
EXPLORATION_PARAMS["tau_OU"]["min"] = 0.4

In [ ]:
grid = make_grid_samples(EXPLORATION_PARAMS, PARAMS_TO_EXPLORE,
                         samples_per_axis=SAMPLES_PER_AXIS, spacing=GRID_SPACING,
                         integer_parameters=INTEGER_PARAMETERS, seed=SAMPLING_SEED,
                         first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX)

# Parametros que has querido cambiar para la simulación
for i, grid_row in enumerate(grid):
    for j, parameters in enumerate(grid_row):
        grid[i][j] = parameters | FIX_PARAMS

In [ ]:
results = simulate_grid(
    grid, 
    stimulus if not USE_STIMULUS_EX else stimulus_ex, 
    time_ms, 
    model_name=MODEL_NAME,
    output_states=["ou_drift"], 
    integer_parameters=INTEGER_PARAMETERS,
    dt_ms=DT_MS, 
    integrator_name=INTEGRATOR_NAME,
    noise_config=NOISE if USE_NOISE else NO_NOISE, 
    noise_seed=NOISE_SEED,
)

for i, row in enumerate(grid):
    for j, parameters in enumerate(row):
        results[i][j] = (results[i][j][0], results[i][j][1] * parameters["weight_noise"])
        

pprint(f'Completed {sum(map(len, results))} simulations.')

In [ ]:
for p in PARAMS_TO_EXPLORE:
    pprint(f'Parameter: {p}')
    pprint(f'  Default: {BASE_PARAMS[p]["default"]}')
    pprint(f'  Min: {BASE_PARAMS[p]["min"]}')
    pprint(f'  Max: {BASE_PARAMS[p]["max"]}')

show_sample_table(grid, list(PARAMS_TO_EXPLORE))

In [ ]:
#Lineas a 1 Hz, 10 Hz

fig, axes = plot_e_grid(
    grid, results, PARAMS_TO_EXPLORE,
    stimulus_ex if USE_STIMULUS_EX else stimulus,
    log_scale=True
)
fig.show()

### Membrane properties

In [ ]:
i=0
PARAMS_TO_EXPLORE = GROUPS['Membrane properties'][i:i+2] #+["whatever"]

pprint(f'Membrane properties len: {len(GROUPS["Membrane properties"])}')
pprint(f'Exploring parameters: {PARAMS_TO_EXPLORE}')
pprint(f'Default: {[BASE_PARAMS[p]["default"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Min: {[BASE_PARAMS[p]["min"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Max: {[BASE_PARAMS[p]["max"] for p in PARAMS_TO_EXPLORE]}')

In [ ]:
FIX_PARAMS = {"Q_e": 1.0}
EXPLORATION_PARAMS = deepcopy(BASE_PARAMS)

# EXPLORATION_PARAMS["g_L"]["min"] = 0.1
# EXPLORATION_PARAMS["g_L"]["max"] = 100.0

# Change the sampling distribution for Sobol parameters.
# sampling_params = {name: EXPLORATION_PARAMS[name] for name in PARAMS_TO_EXPLORE}
# EXPLORATION_PARAMS |= add_beta_config_safe(sampling_params, lambda_=5)
# EXPLORATION_PARAMS |= add_truncnorm_config_safe(sampling_params, std_factor=0.2)

GRID_SPACING = 'linear' # 'log' for logarithmic spacing.

USE_NOISE = False
USE_STIMULUS_EX = False


In [ ]:
grid = make_grid_samples(EXPLORATION_PARAMS, PARAMS_TO_EXPLORE,
                         samples_per_axis=SAMPLES_PER_AXIS, spacing=GRID_SPACING,
                         integer_parameters=INTEGER_PARAMETERS, seed=SAMPLING_SEED,
                         first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX)

# Parametros que has querido cambiar para la simulación
for i, grid_row in enumerate(grid):
    for j, parameters in enumerate(grid_row):
        grid[i][j] = parameters | FIX_PARAMS
    

In [ ]:
results = simulate_grid(
    grid, 
    stimulus if not USE_STIMULUS_EX else stimulus_ex, 
    time_ms, 
    model_name=MODEL_NAME,
    output_states=OUTPUT_STATES, 
    integer_parameters=INTEGER_PARAMETERS,
    dt_ms=DT_MS, 
    integrator_name=INTEGRATOR_NAME,
    noise_config=NOISE if USE_NOISE else NO_NOISE, 
    noise_seed=NOISE_SEED,
)

pprint(f'Completed {sum(map(len, results))} simulations.')

In [ ]:
for p in PARAMS_TO_EXPLORE:
    pprint(f'Parameter: {p}')
    pprint(f'  Default: {EXPLORATION_PARAMS[p]["default"]}')
    pprint(f'  Min: {EXPLORATION_PARAMS[p]["min"]}')
    pprint(f'  Max: {EXPLORATION_PARAMS[p]["max"]}')

show_sample_table(grid, list(PARAMS_TO_EXPLORE))


In [ ]:
pprint(f'Default: {dict((p, BASE_PARAMS[p]["default"]) for p in PARAMS_TO_EXPLORE)}')

fig, axes = plot_e_grid(
    grid, results, PARAMS_TO_EXPLORE,
    stimulus_ex if USE_STIMULUS_EX else stimulus,
    first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX,
)
fig.show()

### Adaptation

In [ ]:
i=0
PARAMS_TO_EXPLORE = GROUPS['Adaptation'][i:i+2] #+["whatever"]

pprint(f'Adaptation len: {len(GROUPS["Adaptation"])}')
pprint(f'Exploring parameters: {PARAMS_TO_EXPLORE}')
pprint(f'Default: {[BASE_PARAMS[p]["default"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Min: {[BASE_PARAMS[p]["min"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Max: {[BASE_PARAMS[p]["max"] for p in PARAMS_TO_EXPLORE]}')

In [ ]:
FIX_PARAMS = {"Q_e": 1.0}
EXPLORATION_PARAMS = deepcopy(BASE_PARAMS)

# EXPLORATION_PARAMS["b_e"]["min"] = 0.0
# EXPLORATION_PARAMS["b_e"]["max"] = 150.0

# Change the sampling distribution for Sobol parameters.
# sampling_params = {name: EXPLORATION_PARAMS[name] for name in PARAMS_TO_EXPLORE}
# EXPLORATION_PARAMS |= add_beta_config_safe(sampling_params, lambda_=5)
# EXPLORATION_PARAMS |= add_truncnorm_config_safe(sampling_params, std_factor=0.2)

GRID_SPACING = 'linear' # 'log' for logarithmic spacing.

USE_NOISE = False
USE_STIMULUS_EX = False


In [ ]:
grid = make_grid_samples(EXPLORATION_PARAMS, PARAMS_TO_EXPLORE,
                         samples_per_axis=SAMPLES_PER_AXIS, spacing=GRID_SPACING,
                         integer_parameters=INTEGER_PARAMETERS, seed=SAMPLING_SEED,
                         first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX)

# Parametros que has querido cambiar para la simulación
for i, grid_row in enumerate(grid):
    for j, parameters in enumerate(grid_row):
        grid[i][j] = parameters | FIX_PARAMS
    

In [ ]:
results = simulate_grid(
    grid, 
    stimulus if not USE_STIMULUS_EX else stimulus_ex, 
    time_ms, 
    model_name=MODEL_NAME,
    output_states=OUTPUT_STATES, 
    integer_parameters=INTEGER_PARAMETERS,
    dt_ms=DT_MS, 
    integrator_name=INTEGRATOR_NAME,
    noise_config=NOISE if USE_NOISE else NO_NOISE, 
    noise_seed=NOISE_SEED,
)

pprint(f'Completed {sum(map(len, results))} simulations.')

In [ ]:
for p in PARAMS_TO_EXPLORE:
    pprint(f'Parameter: {p}')
    pprint(f'  Default: {EXPLORATION_PARAMS[p]["default"]}')
    pprint(f'  Min: {EXPLORATION_PARAMS[p]["min"]}')
    pprint(f'  Max: {EXPLORATION_PARAMS[p]["max"]}')

show_sample_table(grid, list(PARAMS_TO_EXPLORE))


In [ ]:
pprint(f'Default: {dict((p, BASE_PARAMS[p]["default"]) for p in PARAMS_TO_EXPLORE)}')

fig, axes = plot_e_grid(
    grid, results, PARAMS_TO_EXPLORE,
    stimulus_ex if USE_STIMULUS_EX else stimulus,
    first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX,
)
fig.show()

### Synaptic properties

In [ ]:
i=0
PARAMS_TO_EXPLORE = GROUPS['Synaptic properties'][i:i+2] #+["whatever"]

pprint(f'Synaptic properties len: {len(GROUPS["Synaptic properties"])}')
pprint(f'Exploring parameters: {PARAMS_TO_EXPLORE}')
pprint(f'Default: {[BASE_PARAMS[p]["default"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Min: {[BASE_PARAMS[p]["min"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Max: {[BASE_PARAMS[p]["max"] for p in PARAMS_TO_EXPLORE]}')


In [ ]:
FIX_PARAMS = {}
EXPLORATION_PARAMS = deepcopy(BASE_PARAMS)

# EXPLORATION_PARAMS["E_e"]["min"] = -20.0
# EXPLORATION_PARAMS["E_e"]["max"] = 20.0

# Change the sampling distribution for Sobol parameters.
# sampling_params = {name: EXPLORATION_PARAMS[name] for name in PARAMS_TO_EXPLORE}
# EXPLORATION_PARAMS |= add_beta_config_safe(sampling_params, lambda_=5)
# EXPLORATION_PARAMS |= add_truncnorm_config_safe(sampling_params, std_factor=0.2)

GRID_SPACING = 'linear' # 'log' for logarithmic spacing.

USE_NOISE = False
USE_STIMULUS_EX = False

In [ ]:
grid = make_grid_samples(EXPLORATION_PARAMS, PARAMS_TO_EXPLORE,
                         samples_per_axis=SAMPLES_PER_AXIS, spacing=GRID_SPACING,
                         integer_parameters=INTEGER_PARAMETERS, seed=SAMPLING_SEED,
                         first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX)

# Parametros que has querido cambiar para la simulación
for i, grid_row in enumerate(grid):
    for j, parameters in enumerate(grid_row):
        grid[i][j] = parameters | FIX_PARAMS
    

In [ ]:
results = simulate_grid(
    grid, 
    stimulus if not USE_STIMULUS_EX else stimulus_ex, 
    time_ms, 
    model_name=MODEL_NAME,
    output_states=OUTPUT_STATES, 
    integer_parameters=INTEGER_PARAMETERS,
    dt_ms=DT_MS, 
    integrator_name=INTEGRATOR_NAME,
    noise_config=NOISE if USE_NOISE else NO_NOISE, 
    noise_seed=NOISE_SEED,
)

pprint(f'Completed {sum(map(len, results))} simulations.')

In [ ]:
for p in PARAMS_TO_EXPLORE:
    pprint(f'Parameter: {p}')
    pprint(f'  Default: {EXPLORATION_PARAMS[p]["default"]}')
    pprint(f'  Min: {EXPLORATION_PARAMS[p]["min"]}')
    pprint(f'  Max: {EXPLORATION_PARAMS[p]["max"]}')

show_sample_table(grid, list(PARAMS_TO_EXPLORE))


In [ ]:
pprint(f'Default: {dict((p, BASE_PARAMS[p]["default"]) for p in PARAMS_TO_EXPLORE)}')

fig, axes = plot_e_grid(
    grid, results, PARAMS_TO_EXPLORE,
    stimulus_ex if USE_STIMULUS_EX else stimulus,
    first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX,
)
fig.show()

### Network properties

In [ ]:
i=0
PARAMS_TO_EXPLORE = GROUPS['Network properties'][i:i+2] #+["whatever"]

pprint(f'Network properties len: {len(GROUPS["Network properties"])}')
pprint(f'Exploring parameters: {PARAMS_TO_EXPLORE}')
pprint(f'Default: {[BASE_PARAMS[p]["default"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Min: {[BASE_PARAMS[p]["min"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Max: {[BASE_PARAMS[p]["max"] for p in PARAMS_TO_EXPLORE]}')


In [ ]:
FIX_PARAMS = {"Q_e": 1.0}
EXPLORATION_PARAMS = deepcopy(BASE_PARAMS)

# EXPLORATION_PARAMS["N_tot"]["min"] = 1000
# EXPLORATION_PARAMS["N_tot"]["max"] = 50000

# Change the sampling distribution for Sobol parameters.
# sampling_params = {name: EXPLORATION_PARAMS[name] for name in PARAMS_TO_EXPLORE}
# EXPLORATION_PARAMS |= add_beta_config_safe(sampling_params, lambda_=5)
# EXPLORATION_PARAMS |= add_truncnorm_config_safe(sampling_params, std_factor=0.2)

GRID_SPACING = 'linear' # 'log' for logarithmic spacing.

USE_NOISE = False
USE_STIMULUS_EX = False


In [ ]:
grid = make_grid_samples(EXPLORATION_PARAMS, PARAMS_TO_EXPLORE,
                         samples_per_axis=SAMPLES_PER_AXIS, spacing=GRID_SPACING,
                         integer_parameters=INTEGER_PARAMETERS, seed=SAMPLING_SEED,
                         first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX)

# Parametros que has querido cambiar para la simulación
for i, grid_row in enumerate(grid):
    for j, parameters in enumerate(grid_row):
        grid[i][j] = parameters | FIX_PARAMS
    

In [ ]:
results = simulate_grid(
    grid, 
    stimulus if not USE_STIMULUS_EX else stimulus_ex, 
    time_ms, 
    model_name=MODEL_NAME,
    output_states=OUTPUT_STATES, 
    integer_parameters=INTEGER_PARAMETERS,
    dt_ms=DT_MS, 
    integrator_name=INTEGRATOR_NAME,
    noise_config=NOISE if USE_NOISE else NO_NOISE, 
    noise_seed=NOISE_SEED,
)

pprint(f'Completed {sum(map(len, results))} simulations.')

In [ ]:
for p in PARAMS_TO_EXPLORE:
    pprint(f'Parameter: {p}')
    pprint(f'  Default: {EXPLORATION_PARAMS[p]["default"]}')
    pprint(f'  Min: {EXPLORATION_PARAMS[p]["min"]}')
    pprint(f'  Max: {EXPLORATION_PARAMS[p]["max"]}')

show_sample_table(grid, list(PARAMS_TO_EXPLORE))


In [ ]:
pprint(f'Default: {dict((p, BASE_PARAMS[p]["default"]) for p in PARAMS_TO_EXPLORE)}')

fig, axes = plot_e_grid(
    grid, results, PARAMS_TO_EXPLORE,
    stimulus_ex if USE_STIMULUS_EX else stimulus,
    first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX,
)
fig.show()

### Phenomenological threshold coefficients

In [ ]:
PARAMS_TO_EXPLORE = GROUPS['Phenomenological threshold coefficients'] #+["whatever"]
FIRST_PARAM_INDEX = 1
SECOND_PARAM_INDEX = 1


ONLY_TWO_PARAMS = True

pprint(f'Phenomenological threshold coefficients len: {len(GROUPS["Phenomenological threshold coefficients"])}')
pprint(f'Exploring parameters: {PARAMS_TO_EXPLORE}')
pprint(f'Default: {[BASE_PARAMS[p]["default"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Min: {[BASE_PARAMS[p]["min"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Max: {[BASE_PARAMS[p]["max"] for p in PARAMS_TO_EXPLORE]}')


In [ ]:
FIX_PARAMS = {"Q_e": 1.0}
EXPLORATION_PARAMS = deepcopy(BASE_PARAMS)

# Adjust the coefficient ranges around their defaults.
# EXPLORATION_PARAMS = bounds_from_default(
#     EXPLORATION_PARAMS, PARAMS_TO_EXPLORE, margin=0.10,
# )

# Change the sampling distribution for Sobol parameters.
# sampling_params = {name: EXPLORATION_PARAMS[name] for name in PARAMS_TO_EXPLORE}
# EXPLORATION_PARAMS |= add_beta_config_safe(sampling_params, lambda_=5)
# EXPLORATION_PARAMS |= add_truncnorm_config_safe(sampling_params, std_factor=0.2)

if ONLY_TWO_PARAMS:
    EXPLORATION_PARAMS = bounds_from_default(
        EXPLORATION_PARAMS, PARAMS_TO_EXPLORE, margin=0.0,
    )
    for name, index in zip(PARAMS_TO_EXPLORE, (FIRST_PARAM_INDEX, SECOND_PARAM_INDEX)):
        default = EXPLORATION_PARAMS[name]['default'][index]
        EXPLORATION_PARAMS[name]['min'][index] = default * 0.9
        EXPLORATION_PARAMS[name]['max'][index] = default * 1.1


GRID_SPACING = 'linear' # 'log' for logarithmic spacing.


USE_NOISE = False
USE_STIMULUS_EX = False


In [ ]:
grid = make_grid_samples(EXPLORATION_PARAMS, PARAMS_TO_EXPLORE,
                         samples_per_axis=SAMPLES_PER_AXIS, spacing=GRID_SPACING,
                         integer_parameters=INTEGER_PARAMETERS, seed=SAMPLING_SEED,
                         first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX)

# Parametros que has querido cambiar para la simulación
for i, grid_row in enumerate(grid):
    for j, parameters in enumerate(grid_row):
        grid[i][j] = parameters | FIX_PARAMS
    

In [ ]:
results = simulate_grid(
    grid, 
    stimulus if not USE_STIMULUS_EX else stimulus_ex, 
    time_ms, 
    model_name=MODEL_NAME,
    output_states=OUTPUT_STATES, 
    integer_parameters=INTEGER_PARAMETERS,
    dt_ms=DT_MS, 
    integrator_name=INTEGRATOR_NAME,
    noise_config=NOISE if USE_NOISE else NO_NOISE, 
    noise_seed=NOISE_SEED,
)

pprint(f'Completed {sum(map(len, results))} simulations.')

In [ ]:
for p in PARAMS_TO_EXPLORE:
    pprint(f'Parameter: {p}')
    pprint(f'  Default: {EXPLORATION_PARAMS[p]["default"]}')
    pprint(f'  Min: {EXPLORATION_PARAMS[p]["min"]}')
    pprint(f'  Max: {EXPLORATION_PARAMS[p]["max"]}')

show_sample_table(grid, list(PARAMS_TO_EXPLORE))


In [ ]:
pprint(f'Default: {dict((p, BASE_PARAMS[p]["default"]) for p in PARAMS_TO_EXPLORE)}')

fig, axes = plot_e_grid(
    grid, results, PARAMS_TO_EXPLORE,
    stimulus_ex if USE_STIMULUS_EX else stimulus,
    first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX,
)
fig.show()

### Inhibitory coupling scaling

**NO**: Se aplica en Zerlaut de primer orden y puede ser muy **relevante**.

In [ ]:
PARAMS_TO_EXPLORE = GROUPS['Inhibitory coupling scaling'] #+["whatever"]

pprint(f'Inhibitory coupling scaling len: {len(GROUPS["Inhibitory coupling scaling"])}')
pprint(f'Exploring parameters: {PARAMS_TO_EXPLORE}')
pprint(f'Default: {[BASE_PARAMS[p]["default"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Min: {[BASE_PARAMS[p]["min"] for p in PARAMS_TO_EXPLORE]}')
pprint(f'Max: {[BASE_PARAMS[p]["max"] for p in PARAMS_TO_EXPLORE]}')


In [ ]:
FIX_PARAMS = {"Q_e": 1.0}
EXPLORATION_PARAMS = deepcopy(BASE_PARAMS)

# EXPLORATION_PARAMS["S_i"]["min"] = 0.0
# EXPLORATION_PARAMS["S_i"]["max"] = 2.0

# Change the sampling distribution for Sobol parameters.
# sampling_params = {name: EXPLORATION_PARAMS[name] for name in PARAMS_TO_EXPLORE}
# EXPLORATION_PARAMS |= add_beta_config_safe(sampling_params, lambda_=5)
# EXPLORATION_PARAMS |= add_truncnorm_config_safe(sampling_params, std_factor=0.2)

GRID_SPACING = 'linear' # 'log' for logarithmic spacing.

USE_NOISE = False
USE_STIMULUS_EX = False

# S_i scales long-range coupling in second order; this uncoupled setup has none.


In [ ]:
grid = make_grid_samples(EXPLORATION_PARAMS, PARAMS_TO_EXPLORE,
                         samples_per_axis=SAMPLES_PER_AXIS, spacing=GRID_SPACING,
                         integer_parameters=INTEGER_PARAMETERS, seed=SAMPLING_SEED,
                         first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX)

# Parametros que has querido cambiar para la simulación
for i, grid_row in enumerate(grid):
    for j, parameters in enumerate(grid_row):
        grid[i][j] = parameters | FIX_PARAMS
    

In [ ]:
results = simulate_grid(
    grid, 
    stimulus if not USE_STIMULUS_EX else stimulus_ex, 
    time_ms, 
    model_name=MODEL_NAME,
    output_states=OUTPUT_STATES, 
    integer_parameters=INTEGER_PARAMETERS,
    dt_ms=DT_MS, 
    integrator_name=INTEGRATOR_NAME,
    noise_config=NOISE if USE_NOISE else NO_NOISE, 
    noise_seed=NOISE_SEED,
)

pprint(f'Completed {sum(map(len, results))} simulations.')

In [ ]:
for p in PARAMS_TO_EXPLORE:
    pprint(f'Parameter: {p}')
    pprint(f'  Default: {EXPLORATION_PARAMS[p]["default"]}')
    pprint(f'  Min: {EXPLORATION_PARAMS[p]["min"]}')
    pprint(f'  Max: {EXPLORATION_PARAMS[p]["max"]}')

show_sample_table(grid, list(PARAMS_TO_EXPLORE))


In [ ]:
pprint(f'Default: {dict((p, BASE_PARAMS[p]["default"]) for p in PARAMS_TO_EXPLORE)}')

fig, axes = plot_e_grid(
    grid, results, PARAMS_TO_EXPLORE,
    stimulus_ex if USE_STIMULUS_EX else stimulus,
    first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX,
)
fig.show()

#### Se puede simular $S_i$ con **external_in_ex**


$$
\nu_{e\to i}^{\mathrm{input}} = S_i \cdot C_{\mathrm{0}}  + ...
$$

In [ ]:
def simulate_grid_si(grid, stimulus, time_ms, # *,
                  model_name, output_states,
                  integer_parameters, dt_ms, integrator_name,
                  noise_config, noise_seed):
    results = []
    for i, row in enumerate(grid):
        row_results = []
        for j, parameters in enumerate(row):
            pprint(f'Running simulation: {i}, {j}')
            model = build_model(
                parameters, integer_parameters,
                model_class=model_name, variable_of_interest=output_states,
            )


            simulation_stimulus = stimulus
            if 'S_i' in parameters:
                simulation_stimulus = [
                    (name, values * parameters['S_i']
                     if name in ['external_input_in_ex', 'external_input_in_in'] else values)
                    for name, values in stimulus
                ]
            integrator = build_integrator(
                dt_ms, integrator_class=integrator_name,
                noise_config=noise_config,
            )
            simulator = build_single_region_simulator(model, simulation_stimulus, integrator)
            integrator.noise.random_stream.seed(noise_seed)
            time_sim, raw = simulator.run(n_steps=len(time_ms))[0]
            row_results.append((time_sim, raw[:, 0, 0, 0]))
        results.append(row_results)

    return results


In [ ]:

results = simulate_grid_si(
    grid, 
    stimulus if not USE_STIMULUS_EX else stimulus_ex, 
    time_ms, 
    model_name=MODEL_NAME,
    output_states=OUTPUT_STATES, 
    integer_parameters=INTEGER_PARAMETERS,
    dt_ms=DT_MS, 
    integrator_name=INTEGRATOR_NAME,
    noise_config=NOISE if USE_NOISE else NO_NOISE, 
    noise_seed=NOISE_SEED,
)

pprint(f'Completed {sum(map(len, results))} simulations.')

In [ ]:
for p in PARAMS_TO_EXPLORE:
    pprint(f'Parameter: {p}')
    pprint(f'  Default: {EXPLORATION_PARAMS[p]["default"]}')
    pprint(f'  Min: {EXPLORATION_PARAMS[p]["min"]}')
    pprint(f'  Max: {EXPLORATION_PARAMS[p]["max"]}')

show_sample_table(grid, list(PARAMS_TO_EXPLORE))


In [ ]:
pprint(f'Default: {dict((p, BASE_PARAMS[p]["default"]) for p in PARAMS_TO_EXPLORE)}')

fig, axes = plot_e_grid(
    grid, results, PARAMS_TO_EXPLORE,
    stimulus_ex if USE_STIMULUS_EX else stimulus,
    first_index=FIRST_PARAM_INDEX, second_index=SECOND_PARAM_INDEX,
)
fig.show()